# 01 - Standardization

The below documents the standardization steps that produced the published per-campaign columns. The raw inputs (campaign-specific shapefiles / GeoPackages with heterogeneous column names, CRSs, date formats, and integer-coded labels for some campaigns) are *not* part of this archive. This notebook therefore demonstrates the helper functions and the transformation rules against the already-standardized published file. Running the cells is a no-op on canonical data. Their purpose is to show how the published form was obtained.

The five steps were:

1. **Rename + drop columns** -- each campaign's raw column names were rewritten to a canonical set via a per-campaign dictionary; bookkeeping artefacts were dropped.
2. **Integer-code decoding** -- two campaigns stored `crop_main`, `crop_sec`, `growth_main`, and `growth_sec` as integer codes accompanied by lookup CSVs. The codes were mapped to human-readable strings.
3. **CRS standardization** -- all geometries reprojected to `EPSG:4326`.
4. **Timestamp parsing** -- heterogeneous source date formats were normalized to pandas `datetime64`.
5. **Field-ID assignment** -- the 9-digit `YYYYSNNNN` identifier was generated per row.

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd

from utils import standardize_dataframe, assign_field_ids

## Inputs

We start from the `data/` directory containing the published seasonal subdirectories.

*Note: the published files contain a single layer per GeoPackage. If you adapt this notebook to a multi-layer GeoPackage, pass `layer=` explicitly to `gpd.read_file`.*

In [ ]:
DATA_DIR = Path("../data")
CAMPAIGN = "01_LRS23"
CODE     = "LRS23"

gpkg_path = DATA_DIR / CAMPAIGN / f"CropHype-Fields-Kenya_{CODE}.gpkg"
gdf = gpd.read_file(gpkg_path)
print(f"{CODE}: {len(gdf)} rows, {len(gdf.columns)} columns")
gdf.columns.tolist()

## 1 - Column rename and drop

`utils.standardize_dataframe(df, column_mapping, drop_columns)` takes a configuration pair:

- `column_mapping`: dict mapping each source column name to its canonical target name. Keys not in `df.columns` are silently ignored, so the same shape of configuration can be passed across campaigns.
- `drop_columns`: list of column names to remove after renaming (bookkeeping / shapefile artifacts that should not survive into the published file).

In production a different `COLUMN_MAPPING` was assembled per campaign and applied with this function. Because the published file is already standardized, the demonstration below uses empty configuration and is therefore a no-op as the function signature and behavior are what matter.

In [ ]:
COLUMN_MAPPING = {}   # populate per campaign: {"<raw_name>": "<canonical_name>", ...}
DROP_COLUMNS   = []   # populate per campaign: ["<bookkeeping_col>", ...]

gdf_std = standardize_dataframe(gdf, COLUMN_MAPPING, DROP_COLUMNS)
gdf_std.columns.tolist()

## 2 - Integer-code decoding

Two campaigns delivered crop and growth-stage labels as integer codes accompanied by lookup CSVs of the form

```
1, Maize
2, Beans
3, Sorghum
...
```

The decoding pattern is a simple `Series.map(lookup_dict)`. The cell below shows the mechanics with a synthetic lookup; in production, lookups were loaded from the supplied CSVs and applied to the four label columns (`crop_main`, `crop_sec`, `growth_main`, `growth_sec`).

In [ ]:
# Pattern: load a two-column CSV into a dict, then map.
# In production: lookup = dict(pd.read_csv(LOOKUP_CSV, header=None).values)
example_lookup = {1: "Maize", 2: "Beans", 3: "Sorghum"}

raw_codes = pd.Series([1, 3, 2, 1])
decoded   = raw_codes.map(example_lookup)
decoded.tolist()

## 3 - CRS standardization

All campaign geometries were reprojected to `EPSG:4326` (WGS 84 lat/lon) where needed.

In [ ]:
print("Source CRS:        ", gdf.crs)
gdf_4326 = gdf.to_crs(epsg=4326)
print("After standardize: ", gdf_4326.crs)

## 4 - Timestamp parsing

Source `timestamp` columns arrived in mixed formats: locally-formatted date strings, ISO strings, UTC-aware strings, and one campaign where the timestamp was only embedded in the photo filename. All were parsed to `datetime64`. The published column is already in canonical form, so the cell below verifies the dtype and shows a few values.

In [ ]:
print("dtype:", gdf["timestamp"].dtype)
gdf["timestamp"].head(3).tolist()

For reference, the parsing techniques used per campaign were:

- `pd.to_datetime(series, format="mixed", dayfirst=False)` for heterogeneous string dates.
- `pd.to_datetime(series, utc=True).dt.tz_localize(None)` to strip timezone offsets after conversion to UTC.
- `pd.to_datetime(filename.str.extract(r"_(\d{8})\d+\.")[0], format="%Y%m%d")` for the campaign without a timestamp column; the date was extracted from the photo filename.

## 5 - Field-ID assignment

`utils.assign_field_ids(df, year, season_code)` inserts a `field_id` column as the first column. The identifier is a 9-digit integer `YYYYSNNNN`:

| segment | meaning |
|---|---|
| `YYYY` | four-digit year of the campaign. |
| `S`    | single-digit season code: `0` = long rains, `1` = short rains. |
| `NNNN` | 1-based sequence number within the campaign, zero-padded. |

Example: the 27th field of the 2024 long-rains campaign is `202400027`. IDs are assigned in current row order; sort beforehand if a particular ordering is required.

The cell below drops the existing IDs and reassigns them to demonstrate the function. The reassigned IDs match the published ones because the row order is preserved.

In [ ]:
# strip and re-derive — values should match the published field_id
gdf_no_id = gdf.drop(columns=["field_id"])
gdf_reid  = assign_field_ids(gdf_no_id, year=2023, season_code=0)  # LRS -> 0

matches = (gdf_reid["field_id"].values == gdf["field_id"].values).all()
print(f"Reassigned IDs match published IDs: {matches}")
gdf_reid[["field_id"]].head()